# Stage 3 - second judge on the MAIN outputs (Kaggle **GPU T4 x2 required**, Internet ON)
Inputs: the generate notebook output (gen_main_seed0.zip, gen_main_seed1.zip). Judge A verdicts come from the repo (`results/judged/main`). 4800 judgments at roughly 2-4 s each = about 3-5 h (guess; judge A took 3.2 h). The smoke cell stops early if the judge's output format is unusable. Paste back the outputs of the last two cells. **Untested.**

In [1]:
import os, glob, subprocess, torch, json
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart."
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
TAG = "main"
JUDGE_HF, JUDGE_ID = "NousResearch/Hermes-3-Llama-3.1-8B", "hermes3-8b"   # different model family from judge A (Qwen2.5). Fallback if memory/format problems: "microsoft/Phi-3.5-mini-instruct", "phi35-mini"
GEN_ZIPS = sorted(glob.glob("/kaggle/input/**/gen_main_seed*.zip", recursive=True)); print(GEN_ZIPS)
assert len(GEN_ZIPS) == 2, "add the generate notebook output (gen_main_seed0.zip and gen_main_seed1.zip) as input"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs
for z in GEN_ZIPS: subprocess.run(f"unzip -qo {z} -d runs", shell=True)
!cp -r results/judged/main/* runs/main/          # judge A verdicts (no responses) committed in the repo
!ls runs/main runs/main/seed0 | head -30
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

GPUs: 2 ['Tesla T4', 'Tesla T4']
['/kaggle/input/notebooks/sarrazer/kaggle-10-generate/reins-mini/gen_main_seed0.zip', '/kaggle/input/notebooks/sarrazer/kaggle-10-generate/reins-mini/gen_main_seed1.zip']
/kaggle/working/reins-mini
runs/main:
gate_scores_seed0.csv
gate_scores_seed1.csv
seed0
seed1

runs/main/seed0:
harm_only.jsonl
harm_only.judged_qwen25-7b.jsonl
original.jsonl
original.judged_qwen25-7b.jsonl
original_safe.jsonl
original_safe.judged_qwen25-7b.jsonl
plans.jsonl
plans_safe.jsonl
random_active.jsonl
random_active.judged_qwen25-7b.jsonl
random_any.jsonl
random_any.judged_qwen25-7b.jsonl
refusal_only.jsonl
refusal_only.judged_qwen25-7b.jsonl
reins.jsonl
reins.judged_qwen25-7b.jsonl
reins_safe.jsonl
reins_safe.judged_qwen25-7b.jsonl


In [2]:
# smoke test #2 (after fixing the output format with --json-prefill): 20 outputs, one file.
!rm -f runs/main/seed0/*smoke*
!python -u scripts/11_run_judge.py --run-dir runs/main/seed0 --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID}-smoke2 --hf-model {JUDGE_HF} --json-prefill --only reins.jsonl --limit 20
rows = [json.loads(l) for l in open(f"runs/main/seed0/reins.judged_{JUDGE_ID}-smoke2.jsonl")]
drops = sum(r["verdict"] is None for r in rows)
lab = [r["verdict"]["label"] for r in rows if r["verdict"]]
print("smoke: n", len(rows), "dropped", drops, "harmful label share", sum(lab), "of", len(lab))
assert drops <= 2, "still a format problem: switch to the fallback judge (Phi-3.5-mini) and say so in the write-up"
!rm -f runs/main/seed0/*smoke*

config.json: 100%|█████████████████████████████| 883/883 [00:00<00:00, 2.62MB/s]
tokenizer_config.json: 100%|███████████████| 55.8k/55.8k [00:00<00:00, 95.6MB/s]
tokenizer.json: 100%|██████████████████████| 9.09M/9.09M [00:00<00:00, 68.4MB/s]
special_tokens_map.json: 100%|█████████████████| 444/444 [00:00<00:00, 2.42MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100%|████████| 23.9k/23.9k [00:00<00:00, 62.8MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 4 files:   0%|                                   | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 11.1GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 11.1GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 16.1GB            
Reconstructing (incomplete total...):   5%|▍       |  790MB / 16.1GB,  367MB/s  
Reconstructing (incomplete total...):  10%|

In [3]:
for d in sorted(glob.glob(f"runs/{TAG}/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF} --json-prefill", shell=True, check=True)

judging runs/main/seed0


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:09<00:00, 30.41it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  harm_only.jsonl 1/300  3.1s per judgment
  harm_only.jsonl 2/300  2.7s per judgment
  harm_only.jsonl 3/300  2.6s per judgment
  harm_only.jsonl 5/300  2.5s per judgment
  harm_only.jsonl 10/300  2.4s per judgment
  harm_only.jsonl 15/300  2.4s per judgment
  harm_only.jsonl 20/300  2.4s per judgment
  harm_only.jsonl 25/300  2.4s per judgment
  harm_only.jsonl 30/300  2.4s per judgment
  harm_only.jsonl 35/300  2.4s per judgment
  harm_only.jsonl 40/300  2.4s per judgment
  harm_only.jsonl 45/300  2.4s per judgment
  harm_only.jsonl 50/300  2.4s per judgment
  harm_only.jsonl 55/300  2.4s per judgment
  harm_only.jsonl 60/300  2.4s per judgment
  harm_only.jsonl 65/300  2.4s per judgment
  harm_only.jsonl 70/300  2.4s per judgment
  harm_only.jsonl 75/300  2.4s per judgment
  harm_only.jsonl 80/300  2.4s per judgment
  harm_only.jsonl 85/300  2.4s per judgment
  harm_only.jsonl 90/300  2.4s per judgment
  harm_only.jsonl 95/300  2.4s per judgment
  harm_only.jsonl 100/300  2.4s per 

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:09<00:00, 29.67it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  harm_only.jsonl 1/300  3.1s per judgment
  harm_only.jsonl 2/300  2.7s per judgment
  harm_only.jsonl 3/300  2.6s per judgment
  harm_only.jsonl 5/300  2.5s per judgment
  harm_only.jsonl 10/300  2.5s per judgment
  harm_only.jsonl 15/300  2.5s per judgment
  harm_only.jsonl 20/300  2.4s per judgment
  harm_only.jsonl 25/300  2.4s per judgment
  harm_only.jsonl 30/300  2.4s per judgment
  harm_only.jsonl 35/300  2.4s per judgment
  harm_only.jsonl 40/300  2.4s per judgment
  harm_only.jsonl 45/300  2.4s per judgment
  harm_only.jsonl 50/300  2.4s per judgment
  harm_only.jsonl 55/300  2.4s per judgment
  harm_only.jsonl 60/300  2.4s per judgment
  harm_only.jsonl 65/300  2.4s per judgment
  harm_only.jsonl 70/300  2.4s per judgment
  harm_only.jsonl 75/300  2.4s per judgment
  harm_only.jsonl 80/300  2.4s per judgment
  harm_only.jsonl 85/300  2.4s per judgment
  harm_only.jsonl 90/300  2.4s per judgment
  harm_only.jsonl 95/300  2.4s per judgment
  harm_only.jsonl 100/300  2.4s per 

In [4]:
# criteria J1 (C1-C4 under judge B), then J2/J3 (agreement), then the paired table under judge B
!python scripts/12_evaluate.py --runs runs/{TAG} --judge-id {JUDGE_ID} --size 2B --out results/summary_{TAG}_{JUDGE_ID}.csv
!cp results/criteria.json results/criteria_{JUDGE_ID}.json
!python scripts/16_judge_agreement.py --runs runs/{TAG} --judge-a qwen25-7b --judge-b {JUDGE_ID} --out results/judge_agreement_{JUDGE_ID}.csv
!python scripts/15_paired_analysis.py --runs runs/{TAG} --judge-id {JUDGE_ID} --out results/paired_{TAG}_{JUDGE_ID}.csv | grep -v family

method         seed      n  drop    HRR [95% CI]           SRR    OSR     CR   | paper 2B: HRR SRR OSR CR
original       seed0   300     0   77.7 [72.6, 82.0]     1.3   21.0    0.0   |  88.7   1.7   9.7   0.0
random_any     seed0   300     0   78.7 [73.7, 82.9]     1.3   20.0    0.0   |  87.3   1.7  11.0   0.0
random_active  seed0   300     0   79.3 [74.4, 83.5]     1.3   19.3    0.0   | -
harm_only      seed0   300     0   75.7 [70.5, 80.2]     2.0   22.3    0.0   | -
refusal_only   seed0   300     0   30.7 [25.7, 36.1]    33.7   35.0    0.7   | -
reins          seed0   300     0   24.7 [20.1, 29.8]    40.7   33.3    1.3   |  24.8  43.9  18.6  12.8
reins_gate     seed0   300     0   25.3 [20.7, 30.5]    40.7   32.7    1.3   |  25.6  43.2  18.7  12.6
original       seed1   300     0   77.7 [72.6, 82.0]     1.0   21.3    0.0   |  88.7   1.7   9.7   0.0
random_any     seed1   300     0   80.0 [75.1, 84.1]     1.0   19.0    0.0   |  87.3   1.7  11.0   0.0
random_active  seed1   300     0 

In [5]:
!find runs/main -name "*.judged_{JUDGE_ID}.jsonl" | zip -q judged_{JUDGE_ID}_main.zip -@
!cp results/*{JUDGE_ID}* . 2>/dev/null; ls -la judged_{JUDGE_ID}_main.zip *{JUDGE_ID}*
!rm -rf /tmp/REINS

-rw-r--r-- 1 root root  1264 Nov 20 10:35 criteria_hermes3-8b.json
-rw-r--r-- 1 root root   670 Nov 20 10:35 judge_agreement_hermes3-8b.csv
-rw-r--r-- 1 root root 46848 Nov 20 10:35 judged_hermes3-8b_main.zip
-rw-r--r-- 1 root root 46848 Nov 20 10:35 judged_hermes3-8b_main.zip
-rw-r--r-- 1 root root  1237 Nov 20 10:35 paired_main_hermes3-8b.csv
-rw-r--r-- 1 root root  1595 Nov 20 10:35 summary_main_hermes3-8b.csv
